In [2]:
# Imports
import json
import concurrent.futures
import re
import os
from textwrap import dedent
from statistics import mean
from rich.console import Console
from dotenv import load_dotenv
from anthropic import Anthropic

# our custom functions packaged into python files
from my_chat_utils import (
    add_user_message,
    add_assistant_message,
    chat,
)
from report_utils import generate_prompt_evaluation_report
from prompt_evaluator import PromptEvaluator

In [3]:
# Client Initialization and helper functions
load_dotenv(override=True)

client = Anthropic()
model = "claude-haiku-4-5"
console = Console(force_jupyter=False)

In [4]:
# Create an instance of PromptEvaluator
# Increase `max_concurrent_tasks` for greater concurrency, but beware of rate limit errors!
evaluator = PromptEvaluator(client, model, max_concurrent_tasks=1)

In [4]:
def generate_dataset(
    output_file_name: str = "dataset.json", force_generate: bool = False
):
    # generate dataset if output_file_name does NOT exist
    # OR if force_generate = True
    if force_generate or (not os.path.isfile(output_file_name)):
        console.print(f"[red]Generating new dataset[/red] in {output_file_name}")
        dataset = evaluator.generate_dataset(
            # Describe the purpose or goal of the prompt you're trying to test
            task_description="Write a compact, concise 1 day meal plan for a single athlete",
            # Describe the different inputs that your prompt requires
            prompt_inputs_spec={
                "height": "Athlete's height in cm",
                "weight": "Athlete's weight in kg",
                "goal": "Goal of the athlete",
                "restrictions": "Dietary restrictions of the athlete",
            },
            # Where to write the generated dataset
            output_file=output_file_name,
            # Number of test cases to generate (recommend keeping this low if you're getting rate limit errors)
            num_cases=3,
        )
    else:
        console.print(f"[green]Loading dataset[/green] from {output_file_name}")
        with open(output_file_name, "r") as f:
            dataset = json.load(f)

    return dataset


dataset = generate_dataset()

Loading dataset from dataset.json


In [5]:
console.print(dataset)

[
    {
        'prompt_inputs': {
            'height': '178 cm',
            'weight': '68 kg',
            'goal': 'Marathon training with carbohydrate loading for endurance 
performance',
            'restrictions': 'None'
        },
        'solution_criteria': [
            'Meal plan contains exactly 4 meals (breakfast, lunch, snack, 
dinner) with estimated total calories between 3,000-3,500 kcal',
            'Carbohydrates comprise 55-65% of total macronutrients, with 
protein at 15-20% and fat at 20-25%',
            'Plan is presented in compact format (under 400 tokens) with meal 
timing and brief descriptions'
        ],
        'task_description': 'Write a compact, concise 1 day meal plan for a 
single athlete',
        'scenario': 'Testing with a high-endurance athlete (marathon runner) 
requiring high caloric intake and carbohydrate loading, with specific 
macronutrient ratios'
    },
    {
        'prompt_inputs': {
            'height': '180 cm',
            'weight':

In [6]:
# Define and run the prompt you want to evaluate, returning the raw model output
# This function is executed once for each test case
def run_prompt(prompt_inputs, prompt):
    rendered_prompt = evaluator.render(prompt, prompt_inputs)

    messages = []
    add_user_message(messages, rendered_prompt)
    return chat(client, model, messages)

In [7]:
# Prompt templates to evaluate.
# Placeholders like {height} are substituted per test case (from prompt_inputs)
# inside run_prompt via evaluator.render(prompt, prompt_inputs).

naive_prompt = """
What should this person eat?

- Height: {height}
- Weight: {weight}
- Goal: {goal}
- Dietary restrictions: {restrictions}
"""

# here you are being clearer about the meal plan
# - using direct words, such as "generate"
# - adding clear instructions - it should meet
# the athelete's dietary restriction
clear_and_direct_prompt = """
Generate a one-day meal plan for an athlete that 
meets their dietary restrictions.

- Height: {height}
- Weight: {weight}
- Goal: {goal}
- Dietary restrictions: {restrictions}
"""

# now you are adding guidelines over & above being
# clear & direct
clear_direct_and_specific_prompt = """
Generate a one-day meal plan for an athlete that 
meets their dietary restrictions.

- Height: {height}
- Weight: {weight}
- Goal: {goal}
- Dietary restrictions: {restrictions}

Guidelines:
1. Include accurate daily calorie amount
2. Show protein, fat, and carb amounts
3. Specify when to eat each meal
4. Use only foods that fit restrictions
5. List all portion sizes in grams
6. Keep budget-friendly if mentioned
"""

clear_direct_specific_with_xml = """
Generate a one-day meal plan for an athlete that 
meets their dietary restrictions.

<athlete_information> 
- Height: {height} 
- Weight: {weight} 
- Goal: {goal} 
- Dietary restrictions: {restrictions} 
</athlete_information>

Guidelines:
1. Include accurate daily calorie amount
2. Show protein, fat, and carb amounts
3. Specify when to eat each meal
4. Use only foods that fit restrictions
5. List all portion sizes in grams
6. Keep budget-friendly if mentioned
"""

clear_direct_specific_with_xml_and_examples = """
Generate a one-day meal plan for an athlete that meets their dietary restrictions.

<athlete_information> 
- Height: {height} 
- Weight: {weight} 
- Goal: {goal} 
- Dietary restrictions: {restrictions} 
</athlete_information>

Guidelines:
1. Include accurate daily calorie amount
2. Show protein, fat, and carb amounts
3. Specify when to eat each meal
4. Use only foods that fit restrictions
5. List all portion sizes in grams
6. Keep budget-friendly if mentioned

Here is an example with a sample input and an ideal output:
<sample_input>
height: 170
weight: 70
goal: Maintain fitness and improve cholesterol levels
restrictions: High cholesterol
</sample_input>
<ideal_output>
Here is a one-day meal plan for an athlete aiming to maintain fitness and improve cholesterol levels:

*   **Calorie Target:** Approximately 2500 calories
*   **Macronutrient Breakdown:** Protein (140g), Fat (70g), Carbs (340g)

**Meal Plan:**

*   **Breakfast (7:00 AM):** Oatmeal (80g dry weight) with berries (100g) and walnuts (15g). Skim milk (240g).
    *   Protein: 15g, Fat: 15g, Carbs: 60g
*   **Mid-Morning Snack (10:00 AM):** Apple (150g) with almond butter (30g).
    *   Protein: 7g, Fat: 18g, Carbs: 25g
*   **Lunch (1:00 PM):** Grilled chicken breast (120g) salad with mixed greens (150g), cucumber (50g), tomato (50g), and a light vinaigrette dressing (30g). Whole wheat bread (60g).
    *   Protein: 40g, Fat: 15g, Carbs: 70g
*   **Afternoon Snack (4:00 PM):** Greek yogurt (170g, non-fat) with a banana (120g).
    *   Protein: 20g, Fat: 0g, Carbs: 40g
*   **Dinner (7:00 PM):** Baked salmon (140g) with steamed broccoli (200g) and quinoa (75g dry weight).
    *   Protein: 40g, Fat: 20g, Carbs: 80g
*   **Evening Snack (9:00 PM):** Small handful of almonds (20g).
    *   Protein: 8g, Fat: 12g, Carbs: 15g

This meal plan prioritizes lean protein sources, whole grains, fruits, and vegetables, while limiting saturated and trans fats to support healthy cholesterol levels.
</ideal_output>
This example meal plan is well-structured, provides detailed information on food choices and quantities, and aligns with the athlete's goals and restrictions.
"""

In [36]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    prompt=naive_prompt,
    extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact foods, portions, and timing
    """,
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 2.6666666666666665


In [37]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    prompt=clear_and_direct_prompt,
    extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact foods, portions, and timing
    """,
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 5.666666666666667


In [39]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    prompt=clear_direct_and_specific_prompt,
    extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact foods, portions, and timing
    """,
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 6.333333333333333


In [42]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    prompt=clear_direct_specific_with_xml,
    extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact foods, portions, and timing
    """,
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 6.333333333333333


In [9]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    prompt=clear_direct_specific_with_xml_and_examples,
    extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact foods, portions, and timing
    """,
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 5.666666666666667


## Exercise 

We'll generate a new dataset for this exercise, for which we'll write a new `generate_dataset2()` 
function as below.

We have changed the value of the `task_description`, `prompt_inputs_specs`

In [6]:
def generate_dataset2(
    output_file_name: str = "dataset2.json", num_cases=3, force_generate: bool = False
):
    # generate dataset if output_file_name does NOT exist
    # OR if force_generate = True
    if force_generate or (not os.path.isfile(output_file_name)):
        console.print(
            f"[red]Generating new dataset[/red] in {output_file_name} with {num_cases} test cases"
        )
        dataset = evaluator.generate_dataset(
            # Describe the purpose or goal of the prompt you're trying to test
            task_description="""
            Extract topics out of a passage of text from a scholarly article into a JSON
            array of strings
            """,
            # Describe the different inputs that your prompt requires
            prompt_inputs_spec={
                "content": "One paragraph of text from a scholarly journal written in English"
            },
            # Where to write the generated dataset
            output_file=output_file_name,
            # Number of test cases to generate (recommend keeping this low if you're getting rate limit errors)
            num_cases=num_cases,
        )
    else:
        console.print(f"[green]Loading dataset[/green] from {output_file_name}")
        with open(output_file_name, "r") as f:
            dataset = json.load(f)

    return dataset


dataset2 = generate_dataset2(force_generate=True, num_cases=4)

Generating new dataset in dataset2.json with 4 test cases
Generated 1/4 test cases
Generated 2/4 test cases
Generated 3/4 test cases
Generated 4/4 test cases


In [8]:
console.print(dataset2)

[
    {
        'prompt_inputs': {
            'content': 'Recent advances in neurotechnology have sparked 
profound philosophical debates about consciousness and personal identity. The 
development of brain-computer interfaces (BCIs) enables direct neural signal 
decoding, allowing paralyzed patients to control robotic limbs with thought 
alone. Simultaneously, these technologies raise epistemological questions about
the nature of mind-body interaction and whether artificial neural networks can 
truly replicate human cognition. Furthermore, the ethical implications of 
neural data privacy and cognitive liberty have become central concerns in 
bioethics, as neuroscientists, computer engineers, and philosophers collaborate
to establish regulatory frameworks governing the use of invasive and 
non-invasive neuroimaging techniques in both clinical and consumer 
applications.'
        },
        'solution_criteria': [
            'Identifies and separates topics spanning at least three dist

We now call the `evaluator.run_evaluation`, with changes `dataset_file` and `extra_criteria` and
the `prompt` params.

Here is the prompt to be used:

In [ ]:
# adding guidelines to above prompt
# In the above case, the 3rd output wrapped the array in an object
# with "key_words" key. So we'll add some guidelines

In [21]:
# Define and run the prompt you want to evaluate, returning the raw model output
# This function is executed once for each test case
def run_prompt2(prompt_inputs, prompt):
    rendered_prompt = evaluator.render(prompt, prompt_inputs)

    messages = []
    add_user_message(messages, rendered_prompt)
    return chat(client, model, messages)

Let's start with a very basic prompt as below and run the evaluation & inspect the results.

In [ ]:
# starter prompt
exercise_starter_prompt = """
    What topics are in here?

    {content}
"""

results = evaluator.run_evaluation(
    run_prompt_function=run_prompt2,
    # pass in the correct data file name
    dataset_file="dataset2.json",
    # and we are using the exercise prompt
    prompt=exercise_starter_prompt,
    # extra criteria have changed
    extra_criteria="""
    - Contains a JSON array of strings, containing each topic mentioned in the article.
    - The strings should contain only a topic without any extra commentary.
    - Response should contain the JSON array and nothing else.
    """,
    json_output_file="output2.json",
    html_output_file="output2.html",
)

console.print(f"[yellow]Results:[/yellow]\n {results}")

Graded 1/4 test cases
Graded 2/4 test cases
Graded 3/4 test cases
Graded 4/4 test cases
Average score: 2.25
Results:
 [{'output': '# Topics in This Passage\n\n1. **Neurotechnology** - Recent 
advances and developments in brain-related technology\n\n2. **Brain-Computer 
Interfaces (BCIs)** - Direct neural signal decoding and their 
applications\n\n3. **Medical Applications** - Using BCIs to help paralyzed 
patients control robotic limbs\n\n4. **Philosophy of Mind** - Consciousness, 
personal identity, and mind-body interaction\n\n5. **Artificial 
Intelligence/Neural Networks** - Whether AI can replicate human cognition\n\n6.
**Bioethics** - Ethical implications of neurotechnology use\n\n7. **Neural Data
Privacy** - Concerns about protecting brain data\n\n8. **Cognitive Liberty** - 
The right to mental autonomy and freedom\n\n9. **Neuroimaging** - Invasive and 
non-invasive techniques for studying the brain\n\n10. **Regulation & Policy** -
Establishing frameworks to govern neurotechnolog

As expected, we got a very poor evaluation score.

Now open `output2.html` in your browser and look at the `Score` and `Reasoning` columns of the formatted report. The `Scores` are low (as expected), and the `Reasoning` column tells you exactly why - `catastrophically fails all three mandatory requirements: (1) the output is not a valid JSON array of strings, (2) entries contain extra commentary and formatting rather than pure topic strings, and (3) the response includes substantial non-JSON content.` - Bingo! Something to fix in our prompt.

So here is the new prompt & execution results.

In [ ]:
# clear & direct
exercise_clear_and_direct_prompt = """
   Extract key topics mentioned from a passage of text from a scholarly article 
   into a JSON array of strings
   
   {content}
"""

results = evaluator.run_evaluation(
    run_prompt_function=run_prompt2,
    # pass in the correct data file name
    dataset_file="dataset2.json",
    # and we are using the exercise prompt
    prompt=exercise_clear_and_direct_prompt,
    # extra criteria have changed
    extra_criteria="""
    - Contains a JSON array of strings, containing each topic mentioned in the article.
    - The strings should contain only a topic without any extra commentary.
    - Response should contain the JSON array and nothing else.
    """,
    json_output_file="output2-1.json",
    html_output_file="output2-2.html",
)

console.print(f"[yellow]Results:[/yellow]\n {results}")

Graded 1/4 test cases
Graded 2/4 test cases
Graded 3/4 test cases
Graded 4/4 test cases
Average score: 9
Results:
 [{'output': '```json\n[\n  "neurotechnology",\n  "consciousness",\n  "personal
identity",\n  "brain-computer interfaces",\n  "neural signal decoding",\n  
"mind-body interaction",\n  "artificial neural networks",\n  "human 
cognition",\n  "neural data privacy",\n  "cognitive liberty",\n  "bioethics",\n
"neuroimaging techniques",\n  "regulatory frameworks"\n]\n```', 'test_case': 
{'prompt_inputs': {'content': 'Recent advances in neurotechnology have sparked 
profound philosophical debates about consciousness and personal identity. The 
development of brain-computer interfaces (BCIs) enables direct neural signal 
decoding, allowing paralyzed patients to control robotic limbs with thought 
alone. Simultaneously, these technologies raise epistemological questions about
the nature of mind-body interaction and whether artificial neural networks can 
truly replicate human cogniti

Wow! I didn't expect the score to be so good `9` with just a simple tweak to the prompt. Looks like we don't need any further improvements 😊. 

However, I still opened `output2-1.html` and noticed that 3 of 4 test-cases have scored really well (10, 9 and 8), but one has scored 3 because `..violates a mandatory requirement by wrapping the array in an object with a 'key_topics' key instead of providing a pure JSON array.` So I added guidelines and also enclosed `{content}` in prompt within XML tags <passage_of_text>...</passage_of_text>, just to be more explicit. Let's see how this prompt does.

In [27]:
exercise_clear_direct_with_xml = """
   Extract key topics mentioned from a passage of text from a scholarly article 
   into a JSON array of strings
   
   <passage_of_text>
   {content}
   </passage_of_text>

    Guidelines:
    1. The output should strictly be a JSON array of strings
    2. Do not wrap contents in an object
    3. Extract nothing else, but list of topics.
"""


results = evaluator.run_evaluation(
    run_prompt_function=run_prompt2,
    # pass in the correct data file name
    dataset_file="dataset2.json",
    # and we are using the exercise prompt
    prompt=exercise_clear_direct_with_xml,
    # extra criteria have changed
    extra_criteria="""
    - Contains a JSON array of strings, containing each topic mentioned in the article.
    - The strings should contain only a topic without any extra commentary.
    - Response should contain the JSON array and nothing else.
    """,
    json_output_file="output2-2.json",
    html_output_file="output2-2.html",
)

console.print(f"[yellow]Results:[/yellow]\n {results}")

Graded 1/4 test cases
Graded 2/4 test cases
Graded 3/4 test cases
Graded 4/4 test cases
Average score: 9.25
Results:
 [{'output': '```json\n[\n  "neurotechnology",\n  "consciousness",\n  "personal
identity",\n  "brain-computer interfaces",\n  "neural signal decoding",\n  
"robotic limbs",\n  "mind-body interaction",\n  "artificial neural networks",\n
"human cognition",\n  "neural data privacy",\n  "cognitive liberty",\n  
"bioethics",\n  "neuroimaging techniques",\n  "regulatory frameworks",\n  
"invasive neuroimaging",\n  "non-invasive neuroimaging"\n]\n```', 'test_case': 
{'prompt_inputs': {'content': 'Recent advances in neurotechnology have sparked 
profound philosophical debates about consciousness and personal identity. The 
development of brain-computer interfaces (BCIs) enables direct neural signal 
decoding, allowing paralyzed patients to control robotic limbs with thought 
alone. Simultaneously, these technologies raise epistemological questions about
the nature of mind-body i

We got a slightly better score. Open `output2-2.html` in a browser and you'll notice that all the outputs are clean JSON arrays and test cases scored (9,9,10,9) and output met all the specified criteria.

Guess we'll stop here. An average score of `>9` is excellent!